# Decision Tree Classification

**Objective:** build a Decision Tree Classifier that decides whether to
play tennis, using the same dataset from the slides (slide 71), and
verify by hand that the tree is splitting on the feature with the
highest Information Gain (slides 48-56).

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

## The Dataset

14 days of weather, and whether tennis was played that day.

In [ ]:
df = pd.DataFrame({
    'Outlook':     ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast',
                     'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Humidity':    ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal',
                     'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind':        ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong',
                     'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis':  ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes',
                     'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No'],
})
df

## Step 1: Encode categorical features

Scikit-learn's `DecisionTreeClassifier` only accepts numeric input, so
we one-hot encode the feature columns with `pd.get_dummies`. The
label (`PlayTennis`) stays as-is - the classifier handles a
categorical target directly.

In [ ]:
X = pd.get_dummies(df[['Outlook', 'Humidity', 'Wind']])
y = df['PlayTennis']
X

## Step 2: Train the Decision Tree

We use `criterion='entropy'` so the split rule matches the entropy /
information gain formulas from the slides (a real project would
usually leave this as the default `'gini'`, which is faster to
compute and gives similar results).

In [ ]:
model = DecisionTreeClassifier(criterion='entropy', random_state=42)
model.fit(X, y)

## Step 3: Visualize the tree

Compare this to the flowchart-style diagram on slide 48 - internal
nodes are a feature test, branches are the decision rule, and leaves
are the outcome.

In [ ]:
plt.figure(figsize=(14, 8))
plot_tree(model, feature_names=X.columns, class_names=model.classes_,
          filled=True, rounded=True, fontsize=9)
plt.show()

## Step 4: Verify Information Gain by hand

The slides walk through computing entropy and information gain for
splitting on `Outlook` (slides 50-56). Let's reproduce that
calculation in code and confirm it matches `Entropy(S) = 0.940` and
`Information Gain = 0.246` from slide 56.

In [ ]:
import numpy as np

def entropy(labels):
    counts = labels.value_counts()
    probs = counts / len(labels)
    return -(probs * np.log2(probs)).sum() + 0.0  # +0.0 avoids a -0.0 display for pure groups

entropy_s = entropy(df['PlayTennis'])
print('Entropy(S):', round(entropy_s, 3))

In [ ]:
weighted_entropy = 0
for value, group in df.groupby('Outlook'):
    weight = len(group) / len(df)
    e = entropy(group['PlayTennis'])
    weighted_entropy += weight * e
    print(f'Entropy(Outlook={value}): {round(e, 3)}  (n={len(group)})')

information_gain = entropy_s - weighted_entropy
print()
print('Weighted entropy after split on Outlook:', round(weighted_entropy, 3))
print('Information Gain:', round(information_gain, 3))

This matches the by-hand result on slide 56. `feature_importances_`
below is scikit-learn's own measure of how much each encoded feature
contributed to reducing impurity across the whole tree - the
`Outlook_*` columns should dominate, since `Outlook` was the root
split.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

## Step 5: Predict for a new day

Encode a new observation the same way as the training data, then
predict.

In [ ]:
new_day = pd.DataFrame([{'Outlook': 'Sunny', 'Humidity': 'Normal', 'Wind': 'Weak'}])
new_day_encoded = pd.get_dummies(new_day).reindex(columns=X.columns, fill_value=0)

prediction = model.predict(new_day_encoded)
print('Play tennis?', prediction[0])

## Note: Random Forest

Slide 57 introduces the Random Forest Classifier as an "enhanced
version" of a single decision tree - it trains many trees (100 by
default) on random sub-samples of the data and lets them vote.
Swapping it in takes one line:

```python
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X, y)
```